# 🚀 YOLOv1 from Scratch
This notebook trains a custom YOLOv1 object detector on Pascal VOC 2007.
It features:
- **Pretrained ResNet-34 Backbone** with custom Conv head
- **Discriminative Learning Rates** (10x smaller for backbone, standard for head)
- **Automatic Mixed Precision (AMP FP16)** for fast computation
- **Multi-GPU Acceleration (Dual T4)** via PyTorch DataParallel
- **Training/Validation Loss Curves** & **Interactive Bounding Box Predictions**

In [ ]:
# Cell 1: Clone Repository & Setup Working Directory on Kaggle
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/Razamindset/yolo-from-scratch.git"
REPO_DIR = Path("/kaggle/working/yolo-from-scratch")

# If running inside a Kaggle notebook environment, clone the repo if not already present
if Path("/kaggle/working").exists():
    if not REPO_DIR.exists():
        print(f"Cloning repository from {REPO_URL} into {REPO_DIR}...")
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print(f"Repository already present at {REPO_DIR}")

    # Add repo to sys.path and navigate into it
    if str(REPO_DIR) not in sys.path:
        sys.path.append(str(REPO_DIR))
    %cd {REPO_DIR}
else:
    print("Running in local environment.")

In [ ]:
# Cell 2: Imports & GPU Hardware Check
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

from config import (
    ROOT,
    DATA_DIR,
    CHECKPOINT_FILE,
    DEVICE,
    NUM_GPUS,
    BATCH_SIZE,
    NUM_WORKERS,
    PIN_MEMORY,
    BACKBONE_LR,
    HEAD_LR,
    WEIGHT_DECAY,
    EPOCHS,
    CONF_THRESHOLD,
    NMS_IOU_THRESHOLD,
)
from dataset import VOCYolo, VOC_CLASSES
from model import YOLOv1
from loss import YoloLoss
from utils import cellboxes_to_boxes, non_max_suppression, plot_boxes, save_checkpoint, load_checkpoint

print(f"Active Device: {DEVICE}")
print(f"Available GPUs: {NUM_GPUS}")
if NUM_GPUS > 0:
    for i in range(NUM_GPUS):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# Cell 3: Display Active Configuration (from config.py)
print("=== Active Project Configuration ===")
print(f"Project ROOT:           {ROOT}")
print(f"Dataset Directory:      {DATA_DIR}")
print(f"Checkpoint File:        {CHECKPOINT_FILE}")
print(f"Batch Size:             {BATCH_SIZE}")
print(f"Number of Workers:      {NUM_WORKERS}")
print(f"Backbone Learning Rate: {BACKBONE_LR} (10x smaller for pretrained weights)")
print(f"Head Learning Rate:     {HEAD_LR} (for new detection head)")
print(f"Total Epochs:           {EPOCHS}")

In [ ]:
# Cell 4: DataLoaders
train_dataset = VOCYolo(DATA_DIR, split="train", augment=True)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True
)

val_dataset = VOCYolo(DATA_DIR, split="val", augment=False)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print(f"Loaded {len(train_dataset)} training samples ({len(train_loader)} batches).")
print(f"Loaded {len(val_dataset)} validation samples ({len(val_loader)} batches).")

In [ ]:
# Cell 5: Model & Discriminative Learning Rates Optimizer Setup
model = YOLOv1().to(DEVICE)

# 1. Discriminative Learning Rates Parameter Groups
# 2e-5 for pretrained ResNet backbone, 2e-4 for newly initialized head
optimizer = optim.Adam([
    {"params": model.backbone.parameters(), "lr": BACKBONE_LR},
    {"params": model.head.parameters(),     "lr": HEAD_LR},
], weight_decay=WEIGHT_DECAY)

# 2. Multi-GPU DataParallel wrapping (if 2 GPUs present on Kaggle)
if NUM_GPUS > 1:
    print(f"Enabling DataParallel across {NUM_GPUS} GPUs!")
    model = nn.DataParallel(model)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
criterion = YoloLoss()
scaler = torch.amp.GradScaler("cuda", enabled=("cuda" in DEVICE))

# 3. Optional: Resume Training from Previous Checkpoint
RESUME = False
start_epoch = 1
best_val_loss = float("inf")

if RESUME and CHECKPOINT_FILE.exists():
    print(f"Resuming from {CHECKPOINT_FILE}...")
    load_checkpoint(CHECKPOINT_FILE, model, optimizer)
    ckpt = torch.load(CHECKPOINT_FILE, map_location="cpu")
    start_epoch = ckpt.get("epoch", 0) + 1
    best_val_loss = ckpt.get("loss", float("inf"))
    print(f"Resumed! Starting at epoch {start_epoch} (Best Loss: {best_val_loss:.4f})")
else:
    print("Starting fresh training from epoch 1.")

In [ ]:
# Cell 6: Training and Validation Functions
def train_one_epoch(train_loader, model, optimizer, criterion, scaler, device):
    model.train()
    running_loss = 0.0
    loop = tqdm(train_loader, desc="Training", leave=False)

    for batch_idx, (imgs, targets) in enumerate(loop):
        imgs = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        # 1. Forward Pass with AMP (FP16)
        with torch.amp.autocast(device_type="cuda" if "cuda" in device else "cpu", enabled=("cuda" in device)):
            preds = model(imgs)
            loss = criterion(preds, targets)

        # 2. Backward Pass with Gradient Scaling
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()

        # 3. Gradient Clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)

        # 4. Optimizer Step & Scaler Update
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        loop.set_postfix(
            batch_loss=f"{loss.item():.4f}",
            avg_loss=f"{running_loss / (batch_idx + 1):.4f}"
        )

    return running_loss / len(train_loader)


def validate(val_loader, model, criterion, device):
    model.eval()
    total_val_loss = 0.0

    with torch.no_grad():
        for imgs, targets in tqdm(val_loader, desc="Validation", leave=False):
            imgs = imgs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            with torch.amp.autocast(device_type="cuda" if "cuda" in device else "cpu", enabled=("cuda" in device)):
                preds = model(imgs)
                loss = criterion(preds, targets)

            total_val_loss += loss.item()

    return total_val_loss / len(val_loader)

In [ ]:
# Cell 7: Full Training Loop with History Tracking
history = {
    "train_loss": [],
    "val_loss": [],
    "backbone_lr": [],
    "head_lr": []
}

print(f"=== Starting Training for {EPOCHS - start_epoch + 1} Epochs ===")

for epoch in range(start_epoch, EPOCHS + 1):
    b_lr = optimizer.param_groups[0]["lr"]
    h_lr = optimizer.param_groups[1]["lr"]
    print(f"\n--- Epoch [{epoch}/{EPOCHS}] | Backbone LR: {b_lr:.2e} | Head LR: {h_lr:.2e} ---")

    train_loss = train_one_epoch(train_loader, model, optimizer, criterion, scaler, DEVICE)
    val_loss = validate(val_loader, model, criterion, DEVICE)

    # Reduce learning rate when validation loss plateaus
    scheduler.step(val_loss)

    # Record history
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["backbone_lr"].append(b_lr)
    history["head_lr"].append(h_lr)

    print(f"Epoch {epoch:02d} -> Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Save the best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        raw_model = model.module if isinstance(model, nn.DataParallel) else model
        save_checkpoint({
            "epoch": epoch,
            "state_dict": raw_model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "loss": val_loss
        }, filename=str(CHECKPOINT_FILE))
        print(f"  ==> [NEW BEST] Saved checkpoint with Val Loss: {val_loss:.4f}")

print(f"\n=== Training Finished! Best Checkpoint Saved at: {CHECKPOINT_FILE} ===")

In [ ]:
# Cell 8: Plotting Training and Validation Loss Curves
epochs_range = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(10, 5))
plt.plot(epochs_range, history["train_loss"], label="Training Loss", color="#1f77b4", linewidth=2, marker="o")
plt.plot(epochs_range, history["val_loss"], label="Validation Loss", color="#ff7f0e", linewidth=2, marker="s")
plt.title("YOLOv1 Training & Validation Loss over Epochs", fontsize=14, fontweight="bold")
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("YOLO Loss (MSE)", fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9: Visualizing Predictions on Sample Validation Images
# 1. Load the best saved checkpoint into an evaluation model
eval_model = YOLOv1().to(DEVICE)
load_checkpoint(CHECKPOINT_FILE, eval_model)
eval_model.eval()

# 2. Take a small batch of validation samples
val_iter = iter(val_loader)
sample_imgs, sample_targets = next(val_iter)

num_samples_to_plot = 4
print(f"Plotting predictions for {num_samples_to_plot} validation images...\n")

for i in range(num_samples_to_plot):
    img_tensor = sample_imgs[i:i+1].to(DEVICE)

    with torch.no_grad():
        pred = eval_model(img_tensor)

    # Decode the 7x7x30 output into 98 candidate boxes
    decoded_boxes = cellboxes_to_boxes(pred, device="cpu")[0]

    # Filter overlapping and low-confidence boxes with NMS
    final_boxes = non_max_suppression(
        decoded_boxes,
        iou_threshold=NMS_IOU_THRESHOLD,
        threshold=CONF_THRESHOLD
    )

    print(f"Image {i+1}: Detected {len(final_boxes)} object(s)")
    plot_boxes(sample_imgs[i], final_boxes, class_names=VOC_CLASSES)